## Data Cleaning
The Exploratory Data Analysis starts with cleaning data in order to have reliable insights.


### Prices dataset
In order to do that, the EDA starts by inspecing the PUN index GME dataset.  
For reducing granularity and make time series analysis easier, it is better to integrate period data (2021-2025) in the same table.

In [72]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)

# define initial variable
dataset_path = Path('../dataset/raw')
YEARS = range(2021, 2026)

# create prices dataframe
prices_dfs = [
    pd.read_excel(dataset_path / f'GME-PUNi-{year}.xlsx')
    for year in YEARS
]
prices_data = pd.concat(prices_dfs, ignore_index=True)

# fast check
print(f'''
----Rows x Cols:----
{prices_data.shape}\n
----First rows:----
{prices_data.head()}\n
----Last rows:----
{prices_data.tail()}
''')


----Rows x Cols:----
(43824, 3)

----First rows:----
         Data  Ora      €/MWh
0  01/01/2021    1  50,870000
1  01/01/2021    2  48,190000
2  01/01/2021    3  44,679660
3  01/01/2021    4  42,921930
4  01/01/2021    5  40,391510

----Last rows:----
             Data  Ora       €/MWh
43819  31/12/2025   20  121,000000
43820  31/12/2025   21  113,964990
43821  31/12/2025   22  111,200000
43822  31/12/2025   23  113,565000
43823  31/12/2025   24  106,307500



After integrating all data from 01-01-2021 to 31-12-2025, it is necessary to check NULL values, columns data type and other inconsistencies.

In [73]:
# check what to clean
prices_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Data    43824 non-null  object
 1   Ora     43824 non-null  int64 
 2   €/MWh   43824 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.0+ MB


In [74]:
print(f'''
----Date description----
{prices_data['Data'].describe()}\n
----Hour description----
{prices_data['Ora'].describe()}\n
----Prices----
{prices_data['€/MWh'].describe()}\n
''')


----Date description----
count          43824
unique          1826
top       31/10/2021
freq              25
Name: Data, dtype: object

----Hour description----
count    43824.000000
mean        12.500114
std          6.922463
min          1.000000
25%          6.750000
50%         12.500000
75%         18.250000
max         25.000000
Name: Ora, dtype: float64

----Prices----
count          43824
unique         30224
top       120,000000
freq             185
Name: €/MWh, dtype: object




From the code output, it is noticeable that in each row there are no NULL values, but also that there are other inconsistencies:
1. <code>Data</code> datatype is object instead of datetime;
2. <code>€/MWh</code> data type is object instead of float. Also, in Italy floats use colons and not dots;
3. Hours are 25 instead of 24. This is due to a correction of the hour change in <code>31/10/2021</code>.


In [75]:
## Data Cleaning
# 1. Convert Data in datetime
prices_data['Data'] = pd.to_datetime(prices_data['Data'], dayfirst=True, errors='coerce')

# 2. Convert €/MWh in float
prices_data['€/MWh'] = prices_data['€/MWh'].str.replace(',', '.').astype(float)

# 3. Correct hours
prices_data['Ora'] = prices_data['Ora'].apply(lambda x: 24 if x > 24 else x) 

# create unique datetime
prices_data['datetime'] = prices_data['Data'] + pd.to_timedelta(prices_data['Ora'] - 1, unit='h')

# Rename columns and sort
prices_data = prices_data[['datetime', 'Data', '€/MWh']].rename(
    columns={
        'Data': 'Date',
        '€/MWh': 'Prices'
    }
).sort_values('datetime').reset_index(drop=True)

In [76]:
# final cleaning
prices_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  43824 non-null  datetime64[ns]
 1   Date      43824 non-null  datetime64[ns]
 2   Prices    43824 non-null  float64       
dtypes: datetime64[ns](2), float64(1)
memory usage: 1.0 MB


In [77]:
prices_data['datetime'].agg(['min', 'max'])

min   2021-01-01 00:00:00
max   2025-12-31 23:00:00
Name: datetime, dtype: datetime64[ns]

Now that the Dtype for <code>datetime</code> and <code>Date</code> columns is right, and that the minimum value and maximum value for <code>datetime</code> is consistent, the prices data is ready to be exported in /dataset/clean folder.

In [78]:
from pathlib import Path

output_path = Path('../dataset/clean')

# create folder if non-existent
output_path.mkdir(parents=True, exist_ok=True)

# save in CSV format
prices_data.to_csv(output_path / 'pun-index-clean.csv', index=False)

### Generation dataset

This cleaning process includes:
1. The aggregation of the five files containing TERNA data about generated electricity so that we have the complete time series from 2021 to 2025;
2. The deletetion of rows with NULL values, due to the final message that occupies only 1 or 2 cells on each Excel file, indicating which filter had been applied while manually downloading the data;
3. The assignment of the correct data type to each column;
4. Mtu (Market Time Unit) harmonization, since from October 1st energy market is detailed on quartely basis.  

After data cleaning, the new dataset is expected to be copied in the <code>/dataset/clean</code> folder.


In [79]:
# 1. Aggregate generation data in dataframe
gen_dfs = [
    pd.read_excel(dataset_path / f'TERNA-gen-{year}.xlsx')
    for year in YEARS
]
generation_data = pd.concat(gen_dfs, ignore_index=True)

# fast check
print(f'''
----Rows x Cols:----
{generation_data.shape}\n
----First rows:----
{generation_data.head()}\n
----Last rows:----
{generation_data.tail()}
''')


----Rows x Cols:----
(420593, 3)

----First rows:----
                  Date  Actual Generation    Primary Source
0  2021-12-31 23:00:00             11.420           Thermal
1  2021-12-31 23:00:00              3.460             Hydro
2  2021-12-31 23:00:00              0.640        Geothermal
3  2021-12-31 23:00:00              0.000      Photovoltaic
4  2021-12-31 23:00:00              1.516  Self-consumption

----Last rows:----
                                   Date  Actual Generation Primary Source
420588              2025-01-01 00:00:00               0.60     Geothermal
420589              2025-01-01 00:00:00               0.40           Wind
420590              2025-01-01 00:00:00              14.20        Thermal
420591              2025-01-01 00:00:00               2.68          Hydro
420592  Applied filters:  Year is 2025;                NaN            NaN



In [80]:
print(f'''
{generation_data.info()}\n
----Date description----
{generation_data['Date'].describe()}\n
----Actual energy generation description----
{generation_data['Actual Generation'].describe()}\n
----Source of production----
{generation_data['Primary Source'].describe()}\n
''')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420593 entries, 0 to 420592
Data columns (total 3 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Date               420593 non-null  object 
 1   Actual Generation  420588 non-null  float64
 2   Primary Source     420588 non-null  object 
dtypes: float64(1), object(2)
memory usage: 9.6+ MB

None

----Date description----
count                  420593
unique                  70098
top       2024-10-27 02:00:00
freq                       12
Name: Date, dtype: object

----Actual energy generation description----
count    420588.000000
mean          5.083124
std           6.004252
min          -0.010000
25%           0.630000
50%           2.960000
75%           6.560000
max          35.500000
Name: Actual Generation, dtype: float64

----Source of production----
count      420588
unique          6
top       Thermal
freq        70098
Name: Primary Source, dtype: object




In [82]:
# 2. Delete n=5 Rows with NULL values it's the row at the end of each dataset (for each year) which explains the applied filter
generation_data = generation_data.dropna(axis=0, how='any')
print(generation_data.head(10) )       # check

# 3. Convert Date from Object type to Datetime type
generation_data['Date'] = pd.to_datetime(generation_data['Date'], dayfirst=True, errors='coerce')
#    Convert Primary Source from Object to String
generation_data['Primary Source'] = generation_data['Primary Source'].astype('string')

generation_data.columns

                     Actual Generation    Primary Source
Date                                                    
2021-12-31 23:00:00             11.420           Thermal
2021-12-31 23:00:00              3.460             Hydro
2021-12-31 23:00:00              0.640        Geothermal
2021-12-31 23:00:00              0.000      Photovoltaic
2021-12-31 23:00:00              1.516  Self-consumption
2021-12-31 23:00:00              1.820              Wind
2021-12-31 22:00:00              3.720             Hydro
2021-12-31 22:00:00              1.476  Self-consumption
2021-12-31 22:00:00              0.000      Photovoltaic
2021-12-31 22:00:00              0.640        Geothermal


KeyError: 'Date'

In [ ]:

# 4. MTU harmonization
generation_data = generation_data.set_index('Date')
generation_data = (
    generation_data
    .groupby(['Primary Source', pd.Grouper(key='Date', freq='H')])
    .sum()
    .reset_index()
)

generation_pivot = generation_data.pivot(
    index='Date',
    columns='Primary Source',
    values='actual generation'
)
# TODO: Export cleaned copy in folder

# TODO: Export cleaned copy of energy producted totally each hour (no source filter)
